# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# Kather-to-CRC common-seven external classification

This notebook implements the storage-efficient external study that does not
require NCT-CRC-HE-100K. Models are trained only on Kather-5K and CRC-VAL remains
an untouched external test set.

The taxonomy was frozen in notebook 11:

- Kather complex stroma is excluded from training;
- CRC smooth muscle is excluded from evaluation;
- CRC debris and mucus share the Kather debris/mucus target;
- all other classes map one-to-one into seven common labels.

This is **taxonomy-harmonized external transfer**, not native nine-class CRC-VAL
validation. No CRC prediction is used for model selection. Full-data epoch counts
and learning-rate schedules are fixed in advance from the completed Kather work.


In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import time

import pandas as pd
import torch
from IPython.display import display


def locate_project_root():
    return _PUBLICATION_ROOT


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

KATHER_DIR = (
    PROJECT_ROOT / 'Colorectal Histology MNIST'
    / 'Kather_texture_2016_image_tiles_5000'
    / 'Kather_texture_2016_image_tiles_5000'
)
CRC_DIR = PROJECT_ROOT / 'CRC-VAL-HE-7K'
PREFLIGHT_DIR = PROJECT_ROOT / 'artifacts' / 'crc_val_external' / 'preflight'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'crc_val_external' / 'common_seven'
CLASSIFICATION_DIR = OUTPUT_DIR / 'classification'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
FEATURE_DIR = OUTPUT_DIR / 'features'
for directory in (OUTPUT_DIR, CLASSIFICATION_DIR, CHECKPOINT_DIR, FEATURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SEEDS = (11, 89, 181)
MODEL_FILTER = tuple(
    value.strip()
    for value in os.environ.get('CRC_MODELS', 'ResNet18,DINOv2,UNI').split(',')
    if value.strip()
)
FORCE_RETRAIN = False
FORCE_FEATURE_REBUILD = False
NUM_WORKERS = 2

RESNET_EPOCHS = 12
TRANSFORMER_HEAD_EPOCHS = 8
RESNET_BATCH_SIZE = 32
TRANSFORMER_IMAGE_BATCH_SIZE = 16
TRANSFORMER_FEATURE_BATCH_SIZE = 256

print('Project:', PROJECT_ROOT)
print('Device:', DEVICE)
print('Models:', MODEL_FILTER)
print('Seeds:', SEEDS)
print('Outputs:', OUTPUT_DIR)


## Preflight and locked manifests

The external inventory must have been created by notebook 11. The assertions
below prevent accidental inclusion of smooth muscle or Kather complex stroma.
Patient IDs remain missing unless an official manifest was supplied in notebook
11; filenames are never interpreted as patients.


In [ ]:
from Methods.CRCExternalValidation import (
    COMMON_CLASS_NAMES,
    build_crc_common_manifest,
    build_kather_common_manifest,
    classification_tables,
    extract_transformer_cache,
    train_evaluate_resnet,
    train_evaluate_transformer_heads,
)
from Methods.DINOv2Attribution import (
    build_dinov2_classifier,
    resolve_dinov2_transform,
)
from Methods.UNIAttribution import build_uni_classifier, resolve_uni_transform

readiness_path = PREFLIGHT_DIR / 'preflight_readiness.json'
inventory_path = PREFLIGHT_DIR / 'crc_val_inventory_with_metadata.csv'
if not readiness_path.is_file() or not inventory_path.is_file():
    raise FileNotFoundError('Run notebook 11 before external classification')

readiness = json.loads(readiness_path.read_text(encoding='utf-8'))
assert readiness['dataset_images'] == 7180
assert readiness['dataset_classes'] == 9
assert readiness['external_predictions_examined'] is False

kather_manifest = build_kather_common_manifest(KATHER_DIR)
crc_manifest = build_crc_common_manifest(CRC_DIR, inventory_path)
assert len(kather_manifest) == 4375
assert len(crc_manifest) == 6588
assert kather_manifest['class_name'].nunique() == 7
assert crc_manifest['class_name'].nunique() == 7
assert '03_COMPLEX' not in set(kather_manifest['source_class'])
assert 'MUS' not in set(crc_manifest['source_class'])

kather_manifest.to_csv(OUTPUT_DIR / 'kather_common_seven_training_manifest.csv', index=False)
crc_manifest.to_csv(OUTPUT_DIR / 'crc_common_seven_external_manifest.csv', index=False)
display(kather_manifest.groupby(['class_name', 'source_class']).size().rename('images'))
display(crc_manifest.groupby(['class_name', 'source_class']).size().rename('images'))
print('Inference level:', readiness['inference_level'])


## Fully fine-tuned ResNet18

ResNet18 is initialized from ImageNet weights and all layers are fine-tuned for 12
fixed epochs on all eligible Kather images. CRC images are resized to the 150-pixel
Kather training size before ImageNet normalization. A cosine learning-rate schedule
replaces validation-driven early stopping so CRC-VAL cannot influence training.


In [ ]:
resnet_prediction_path = CLASSIFICATION_DIR / 'resnet18_external_predictions.csv'
resnet_history_path = CLASSIFICATION_DIR / 'resnet18_training_history.csv'
if 'ResNet18' in MODEL_FILTER:
    started = time.time()
    resnet_predictions, resnet_history = train_evaluate_resnet(
        kather_manifest,
        crc_manifest,
        DEVICE,
        CHECKPOINT_DIR / 'resnet18',
        seeds=SEEDS,
        epochs=RESNET_EPOCHS,
        learning_rate=1e-4,
        weight_decay=1e-4,
        batch_size=RESNET_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        force_retrain=FORCE_RETRAIN,
    )
    resnet_predictions.to_csv(resnet_prediction_path, index=False)
    if not resnet_history.empty:
        resnet_history.to_csv(resnet_history_path, index=False)
    (CLASSIFICATION_DIR / 'resnet18.complete').write_text(
        'predictions and all seed checkpoints complete\n', encoding='utf-8'
    )
    print(f'ResNet18 hours: {(time.time() - started) / 3600:.2f}')
else:
    resnet_predictions = (
        pd.read_csv(resnet_prediction_path)
        if resnet_prediction_path.is_file() else pd.DataFrame()
    )


## Frozen DINOv2

The DINOv2 encoder remains frozen. Deterministic CLS embeddings are cached once
for Kather and CRC-VAL; only the seven-class lightweight head is trained for eight
fixed epochs per seed.


In [ ]:
dino_prediction_path = CLASSIFICATION_DIR / 'dinov2_external_predictions.csv'
dino_history_path = CLASSIFICATION_DIR / 'dinov2_training_history.csv'
if 'DINOv2' in MODEL_FILTER:
    started = time.time()
    dino_model = build_dinov2_classifier(len(COMMON_CLASS_NAMES), DEVICE)
    dino_transform, dino_config = resolve_dinov2_transform(dino_model.encoder)
    dino_preprocessing_id = 'DINOv2|224|timm_pretrained_cfg'
    dino_train_cache = extract_transformer_cache(
        dino_model,
        kather_manifest,
        DEVICE,
        dino_transform,
        FEATURE_DIR / 'dinov2_kather_common_seven_float32.pt',
        dino_preprocessing_id,
        batch_size=TRANSFORMER_IMAGE_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        overwrite=FORCE_FEATURE_REBUILD,
    )
    dino_external_cache = extract_transformer_cache(
        dino_model,
        crc_manifest,
        DEVICE,
        dino_transform,
        FEATURE_DIR / 'dinov2_crc_common_seven_float32.pt',
        dino_preprocessing_id,
        batch_size=TRANSFORMER_IMAGE_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        overwrite=FORCE_FEATURE_REBUILD,
    )
    dino_predictions, dino_history = train_evaluate_transformer_heads(
        dino_model,
        dino_train_cache,
        dino_external_cache,
        crc_manifest,
        DEVICE,
        CHECKPOINT_DIR / 'dinov2',
        model_name='DINOv2',
        seeds=SEEDS,
        epochs=TRANSFORMER_HEAD_EPOCHS,
        learning_rate=1e-3,
        weight_decay=1e-4,
        batch_size=TRANSFORMER_FEATURE_BATCH_SIZE,
        force_retrain=FORCE_RETRAIN,
    )
    dino_predictions.to_csv(dino_prediction_path, index=False)
    if not dino_history.empty:
        dino_history.to_csv(dino_history_path, index=False)
    (CLASSIFICATION_DIR / 'dinov2.complete').write_text(
        'predictions and all seed checkpoints complete\n', encoding='utf-8'
    )
    del dino_model, dino_train_cache, dino_external_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'DINOv2 hours: {(time.time() - started) / 3600:.2f}')
else:
    dino_predictions = (
        pd.read_csv(dino_prediction_path)
        if dino_prediction_path.is_file() else pd.DataFrame()
    )


## Frozen UNI

UNI uses the same lightweight-head protocol as DINOv2. Set `UNI_ASSETS_DIR` in the
environment when the local UNI assets should be used instead of Hugging Face.


In [ ]:
uni_prediction_path = CLASSIFICATION_DIR / 'uni_external_predictions.csv'
uni_history_path = CLASSIFICATION_DIR / 'uni_training_history.csv'
if 'UNI' in MODEL_FILTER:
    started = time.time()
    uni_model = build_uni_classifier(
        PROJECT_ROOT,
        len(COMMON_CLASS_NAMES),
        DEVICE,
        assets_dir=os.environ.get('UNI_ASSETS_DIR') or None,
    )
    uni_transform, uni_config = resolve_uni_transform(uni_model.encoder)
    uni_preprocessing_id = 'UNI|224|timm_pretrained_cfg'
    uni_train_cache = extract_transformer_cache(
        uni_model,
        kather_manifest,
        DEVICE,
        uni_transform,
        FEATURE_DIR / 'uni_kather_common_seven_float32.pt',
        uni_preprocessing_id,
        batch_size=TRANSFORMER_IMAGE_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        overwrite=FORCE_FEATURE_REBUILD,
    )
    uni_external_cache = extract_transformer_cache(
        uni_model,
        crc_manifest,
        DEVICE,
        uni_transform,
        FEATURE_DIR / 'uni_crc_common_seven_float32.pt',
        uni_preprocessing_id,
        batch_size=TRANSFORMER_IMAGE_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        overwrite=FORCE_FEATURE_REBUILD,
    )
    uni_predictions, uni_history = train_evaluate_transformer_heads(
        uni_model,
        uni_train_cache,
        uni_external_cache,
        crc_manifest,
        DEVICE,
        CHECKPOINT_DIR / 'uni',
        model_name='UNI',
        seeds=SEEDS,
        epochs=TRANSFORMER_HEAD_EPOCHS,
        learning_rate=1e-3,
        weight_decay=1e-4,
        batch_size=TRANSFORMER_FEATURE_BATCH_SIZE,
        force_retrain=FORCE_RETRAIN,
    )
    uni_predictions.to_csv(uni_prediction_path, index=False)
    if not uni_history.empty:
        uni_history.to_csv(uni_history_path, index=False)
    (CLASSIFICATION_DIR / 'uni.complete').write_text(
        'predictions and all seed checkpoints complete\n', encoding='utf-8'
    )
    del uni_model, uni_train_cache, uni_external_cache
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'UNI hours: {(time.time() - started) / 3600:.2f}')
else:
    uni_predictions = (
        pd.read_csv(uni_prediction_path)
        if uni_prediction_path.is_file() else pd.DataFrame()
    )


## External classification results

Every model-seed combination must contain one prediction for each of the 6,588
eligible CRC tiles and all seven common true classes. Seeds measure optimization
variability; they are not independent patient samples.


In [ ]:
prediction_frames = [
    frame for frame in (resnet_predictions, dino_predictions, uni_predictions)
    if not frame.empty
]
if not prediction_frames:
    raise RuntimeError('No model predictions are available')
predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_csv(CLASSIFICATION_DIR / 'all_external_predictions_and_logits.csv', index=False)

for (model, seed), frame in predictions.groupby(['model', 'seed']):
    assert len(frame) == 6588, (model, seed, len(frame))
    assert frame['relative_path'].nunique() == 6588
    assert frame['class_name'].nunique() == 7
    assert 'MUS' not in set(frame['source_class'])

overall, per_class, confusion = classification_tables(predictions)
overall.to_csv(CLASSIFICATION_DIR / 'classification_per_seed.csv', index=False)
per_class.to_csv(CLASSIFICATION_DIR / 'classification_per_class.csv', index=False)
confusion.to_csv(CLASSIFICATION_DIR / 'confusion_matrices_long.csv', index=False)
source_class_results = (
    predictions.assign(source_class_correct=lambda frame: frame['correct'].astype(float))
    .groupby(['model', 'seed', 'source_class'], as_index=False)
    .agg(
        images=('relative_path', 'nunique'),
        accuracy=('source_class_correct', 'mean'),
    )
)
source_class_results.to_csv(
    CLASSIFICATION_DIR / 'classification_by_original_crc_class.csv', index=False
)
crc_manifest.groupby(['class_name', 'source_class']).size().rename('images').reset_index().to_csv(
    CLASSIFICATION_DIR / 'external_class_counts.csv', index=False
)

summary = (
    overall.groupby('model')[['accuracy', 'balanced_accuracy', 'macro_f1']]
    .agg(['mean', 'std'])
    .round(4)
)
display(summary)
display(
    per_class.groupby(['model', 'class_name'])[['recall', 'f1']]
    .mean().round(4)
)
display(
    source_class_results.groupby(['model', 'source_class'])['accuracy']
    .mean().round(4)
)


## Completion record

These results are external tile-classification estimates under the common-seven
taxonomy. Patient-cluster intervals remain unavailable unless notebook 11 joined
an official tile-to-patient manifest. Notebook 13 will use the already frozen
prediction-independent 98-image cohort for Grad-CAM, gradient-weighted rollout,
patch occlusion, and scale-free deletion evaluation.


In [ ]:
completed_models = sorted(predictions['model'].unique())
metadata = {
    'study': 'Kather-to-CRC common-seven external transfer',
    'models': completed_models,
    'seeds': sorted(int(value) for value in predictions['seed'].unique()),
    'kather_training_images': int(len(kather_manifest)),
    'crc_external_images': int(len(crc_manifest)),
    'common_classes': list(COMMON_CLASS_NAMES),
    'excluded_kather_class': '03_COMPLEX',
    'excluded_crc_class': 'MUS',
    'merged_crc_classes': {'DEB': 'debris_mucus', 'MUC': 'debris_mucus'},
    'inference_level': readiness['inference_level'],
    'patient_metadata_available': readiness['patient_metadata_available'],
    'native_nine_class_claim_supported': False,
    'biological_causality_claim_supported': False,
}
(OUTPUT_DIR / 'classification_run_metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)
display(pd.Series(metadata, name='value').to_frame())
